# Modelo numero 1


In [ ]:
import os
import numpy as np
from PIL import Image
from torchvision import transforms
import torch

# Directorio donde se encuentran las imágenes
data_dir = 'C:/Users/estud/OneDrive/Escritorio/BASE IA/BASE DE DATOS'

# Transformaciones para normalizar las imágenes
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Cargar las imágenes desde subcarpetas y aplicar transformaciones
def load_images(data_dir):
    images = []
    for subdir, dirs, files in os.walk(data_dir):
        for filename in files:
            if filename.endswith(".jpg") or filename.endswith(".png"):
                img_path = os.path.join(subdir, filename)
                image = Image.open(img_path)
                image = transform(image)
                images.append(image)
                print(f"Loaded {filename} from {subdir}")
    if not images:
        print("No images found in directory.")
    return torch.stack(images) if images else torch.tensor([])

X_train = load_images(data_dir)

if X_train.size(0) == 0:
    print("No images were loaded. Please check your directory and file types.")
else:
  1
  #print(f"Loaded {X_train.size(0)} images.")


Loaded exswc (1).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (3).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (11).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (9).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (13).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (5).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (10).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (4).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (6).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (7).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (2).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (8).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (12).jpg from /content/d

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, 0.8),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, 0.8),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, 0.8),
            nn.Linear(1024, 300 * 300),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 1, 300, 300)
        return img

generator = Generator()


In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(300 * 300, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

discriminator = Discriminator()


In [ ]:
import torch.optim as optim

adversarial_loss = torch.nn.BCELoss()

optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))


In [ ]:
import torchvision.utils as vutils

num_epochs = 200
batch_size = 64
sample_interval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator.to(device)
discriminator.to(device)
X_train = X_train.to(device)

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        # Preparar datos reales
        real_imgs = X_train[i:i+batch_size]
        valid = torch.ones(real_imgs.size(0), 1, device=device, dtype=torch.float32)
        fake = torch.zeros(real_imgs.size(0), 1, device=device, dtype=torch.float32)

        # Generar imágenes falsas
        z = torch.randn(real_imgs.size(0), 100, device=device)
        gen_imgs = generator(z)

        # Entrenar el discriminador
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = 0.5 * (real_loss + fake_loss)
        d_loss.backward()
        optimizer_D.step()

        # Entrenar el generador
        optimizer_G.zero_grad()
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

    print(f"[Epoch {epoch}/{num_epochs}] [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")

    # Guardar imágenes de muestra
    if epoch % sample_interval == 0:
        vutils.save_image(gen_imgs.data[:25], f"drive/MyDrive/Habla/epoch_{epoch}.png", nrow=5, normalize=True)


[Epoch 0/200] [D loss: 0.7218475341796875] [G loss: 0.36061543226242065]
[Epoch 1/200] [D loss: 0.33686065673828125] [G loss: 0.8661213517189026]
[Epoch 2/200] [D loss: 1.1543608903884888] [G loss: 0.1355467438697815]
[Epoch 3/200] [D loss: 0.6702326536178589] [G loss: 0.38645046949386597]
[Epoch 4/200] [D loss: 0.5746883153915405] [G loss: 0.3548731803894043]
[Epoch 5/200] [D loss: 0.7184740900993347] [G loss: 0.37565916776657104]
[Epoch 6/200] [D loss: 0.5727686285972595] [G loss: 0.41612517833709717]
[Epoch 7/200] [D loss: 0.843035876750946] [G loss: 0.14469218254089355]
[Epoch 8/200] [D loss: 0.5992463827133179] [G loss: 0.42045751214027405]
[Epoch 9/200] [D loss: 0.5797050595283508] [G loss: 0.45809561014175415]
[Epoch 10/200] [D loss: 0.7973489165306091] [G loss: 0.2874694764614105]
[Epoch 11/200] [D loss: 0.4685336649417877] [G loss: 0.7375127077102661]
[Epoch 12/200] [D loss: 0.4669150114059448] [G loss: 0.5945858955383301]
[Epoch 13/200] [D loss: 0.421085000038147] [G loss: 0.

# MODELO 1 MEJORADO

In [ ]:
import os
import numpy as np
from PIL import Image
from torchvision import transforms
import torch

data_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS'

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def load_images(data_dir):
    images = []
    for subdir, dirs, files in os.walk(data_dir):
        for filename in files:
            if filename.endswith(".jpg") or filename.endswith(".png"):
                img_path = os.path.join(subdir, filename)
                image = Image.open(img_path)
                image = transform(image)
                images.append(image)
                print(f"Loaded {filename} from {subdir}")
    if not images:
        print("No images found in directory.")
    return torch.stack(images) if images else torch.tensor([])

X_train = load_images(data_dir)

if X_train.size(0) == 0:
    print("No images were loaded. Please check your directory and file types.")
else:
  1
  #print(f"Loaded {X_train.size(0)} images.")


Loaded exswc (1).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (3).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (11).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (9).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (13).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (5).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (10).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (4).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (6).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (7).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (2).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (8).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (12).jpg from /content/d

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, 0.8),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, 0.8),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, 0.8),
            nn.Linear(1024, 2048),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(2048, 0.8),
            nn.Linear(2048, 300 * 300),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 1, 300, 300)
        return img

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(300 * 300, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

generator = Generator()
discriminator = Discriminator()


In [ ]:
import torch.optim as optim

adversarial_loss = torch.nn.BCELoss()

optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))


In [ ]:
import torchvision.utils as vutils

num_epochs = 500  # Aumentando las épocas para un entrenamiento más profundo
batch_size = 64
sample_interval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator.to(device)
discriminator.to(device)
X_train = X_train.to(device)

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        real_imgs = X_train[i:i+batch_size]
        valid = torch.ones(real_imgs.size(0), 1, device=device, dtype=torch.float32)
        fake = torch.zeros(real_imgs.size(0), 1, device=device, dtype=torch.float32)

        z = torch.randn(real_imgs.size(0), 100, device=device)
        gen_imgs = generator(z)

        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = 0.5 * (real_loss + fake_loss)
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

    if epoch % sample_interval == 0:
        vutils.save_image(gen_imgs.data[:25], f"drive/MyDrive/Habla/epoch_{epoch}.png", nrow=5, normalize=True)
        print(f"[Epoch {epoch}/{num_epochs}] [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")

    # Control de pérdida de modelos para mejorar la precisión
    if d_loss.item() < 0.5:
        optimizer_D.param_groups[0]['lr'] /= 2
        print(f"Discriminator learning rate reduced to {optimizer_D.param_groups[0]['lr']}")
    if g_loss.item() > 1.5:
        optimizer_G.param_groups[0]['lr'] /= 2
        print(f"Generator learning rate reduced to {optimizer_G.param_groups[0]['lr']}")


[Epoch 0/500] [D loss: 0.5611610412597656] [G loss: 0.42955076694488525]
Discriminator learning rate reduced to 5e-05
Discriminator learning rate reduced to 2.5e-05
Discriminator learning rate reduced to 1.25e-05
Discriminator learning rate reduced to 6.25e-06
Discriminator learning rate reduced to 3.125e-06
Discriminator learning rate reduced to 1.5625e-06
Discriminator learning rate reduced to 7.8125e-07
Discriminator learning rate reduced to 3.90625e-07
Discriminator learning rate reduced to 1.953125e-07
[Epoch 10/500] [D loss: 0.5181512832641602] [G loss: 0.4351324737071991]
[Epoch 20/500] [D loss: 0.7840311527252197] [G loss: 0.287835955619812]
[Epoch 30/500] [D loss: 0.7456238269805908] [G loss: 0.35206472873687744]
[Epoch 40/500] [D loss: 0.9145918488502502] [G loss: 0.3923933506011963]
[Epoch 50/500] [D loss: 0.6983184218406677] [G loss: 0.40024471282958984]
[Epoch 60/500] [D loss: 0.8494094014167786] [G loss: 0.4839189648628235]
[Epoch 70/500] [D loss: 0.8757981061935425] [G l

# MODELO 1 MEJORADO 2

In [ ]:
import os
import numpy as np
from PIL import Image
from torchvision import transforms
import torch

data_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS'

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def load_images(data_dir):
    images = []
    for subdir, dirs, files in os.walk(data_dir):
        for filename in files:
            if filename.endswith(".jpg") or filename.endswith(".png"):
                img_path = os.path.join(subdir, filename)
                image = Image.open(img_path)
                image = transform(image)
                images.append(image)
                print(f"Loaded {filename} from {subdir}")
    if not images:
        print("No images found in directory.")
    return torch.stack(images) if images else torch.tensor([])

X_train = load_images(data_dir)

if X_train.size(0) == 0:
    print("No images were loaded. Please check your directory and file types.")
else:
    print(f"Loaded {X_train.size(0)} images.")


Loaded exswc (1).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (3).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (11).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (9).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (13).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (5).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (10).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (4).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (6).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (7).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (2).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (8).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (12).jpg from /content/d

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, 0.8),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, 0.8),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024, 0.8),
            nn.Linear(1024, 300 * 300),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 1, 300, 300)
        return img

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(300 * 300, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

generator = Generator()
discriminator = Discriminator()


In [ ]:
import torch.optim as optim

adversarial_loss = torch.nn.BCELoss()

optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.00005, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.00005, betas=(0.5, 0.999))


In [ ]:
import torchvision.utils as vutils

num_epochs = 500
batch_size = 64
sample_interval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator.to(device)
discriminator.to(device)
X_train = X_train.to(device)

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        real_imgs = X_train[i:i+batch_size]
        valid = torch.ones(real_imgs.size(0), 1, device=device, dtype=torch.float32)
        fake = torch.zeros(real_imgs.size(0), 1, device=device, dtype=torch.float32)

        z = torch.randn(real_imgs.size(0), 100, device=device)
        gen_imgs = generator(z)

        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = 0.5 * (real_loss + fake_loss)
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

    if epoch % sample_interval == 0:
        vutils.save_image(gen_imgs.data[:25], f"drive/MyDrive/Habla/epoch_{epoch}.png", nrow=5, normalize=True)
        print(f"[Epoch {epoch}/{num_epochs}] [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")

    # Ajuste dinámico de la tasa de aprendizaje si las pérdidas son muy altas o bajas
    if d_loss.item() < 0.3:
        optimizer_D.param_groups[0]['lr'] *= 1.1
        print(f"Discriminator learning rate increased to {optimizer_D.param_groups[0]['lr']}")
    if g_loss.item() > 1.5:
        optimizer_G.param_groups[0]['lr'] /= 1.5
        print(f"Generator learning rate reduced to {optimizer_G.param_groups[0]['lr']}")


[Epoch 0/500] [D loss: 0.19177643954753876] [G loss: 1.2207814455032349]
Discriminator learning rate increased to 5.500000000000001e-05
Discriminator learning rate increased to 6.0500000000000014e-05
Generator learning rate reduced to 3.3333333333333335e-05
Discriminator learning rate increased to 6.655000000000002e-05
Generator learning rate reduced to 2.2222222222222223e-05
Discriminator learning rate increased to 7.320500000000003e-05
Generator learning rate reduced to 1.4814814814814815e-05
Discriminator learning rate increased to 8.052550000000004e-05
Generator learning rate reduced to 9.876543209876543e-06
Discriminator learning rate increased to 8.857805000000005e-05
Discriminator learning rate increased to 9.743585500000006e-05
Generator learning rate reduced to 6.584362139917696e-06
Discriminator learning rate increased to 0.00010717944050000007
Generator learning rate reduced to 4.38957475994513e-06
Discriminator learning rate increased to 0.00011789738455000008
Generator lea

# MODELO 1 MEJORADO 3

In [ ]:
import os
import numpy as np
from PIL import Image
from torchvision import transforms
import torch

data_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS'

transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def load_images(data_dir):
    images = []
    for subdir, dirs, files in os.walk(data_dir):
        for filename in files:
            if filename.endswith(".jpg") or filename.endswith(".png"):
                img_path = os.path.join(subdir, filename)
                image = Image.open(img_path)
                image = transform(image)
                images.append(image)
                print(f"Loaded {filename} from {subdir}")
    if not images:
        print("No images found in directory.")
    return torch.stack(images) if images else torch.tensor([])

X_train = load_images(data_dir)

if X_train.size(0) == 0:
    print("No images were loaded. Please check your directory and file types.")
else:
    print(f"Loaded {X_train.size(0)} images.")


Loaded exswc (1).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (3).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (11).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (9).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (13).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (5).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (10).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (4).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (6).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (7).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (2).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (8).jpg from /content/drive/MyDrive/BASE DE ESQUEMAS/CAMION DUMPER
Loaded exswc (12).jpg from /content/d

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(128, 0.8),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256, 0.8),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512, 0.8),
            nn.Linear(512, 300 * 300),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 1, 300, 300)
        return img

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(300 * 300, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        validity = self.model(img_flat)
        return validity

generator = Generator()
discriminator = Discriminator()


In [ ]:
import torch.optim as optim

adversarial_loss = torch.nn.BCELoss()

optimizer_G = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))


In [ ]:
import torchvision.utils as vutils

num_epochs = 200
batch_size = 64
sample_interval = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator.to(device)
discriminator.to(device)
X_train = X_train.to(device)

for epoch in range(num_epochs):
    for i in range(0, len(X_train), batch_size):
        real_imgs = X_train[i:i+batch_size]
        valid = torch.ones(real_imgs.size(0), 1, device=device, dtype=torch.float32)
        fake = torch.zeros(real_imgs.size(0), 1, device=device, dtype=torch.float32)

        z = torch.randn(real_imgs.size(0), 100, device=device)
        gen_imgs = generator(z)

        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = 0.5 * (real_loss + fake_loss)
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
        g_loss = adversarial_loss(discriminator(gen_imgs), valid)
        g_loss.backward()
        optimizer_G.step()

    if epoch % sample_interval == 0:
        vutils.save_image(gen_imgs.data[:25], f"drive/MyDrive/Habla/epoch_{epoch}.png", nrow=5, normalize=True)
        print(f"[Epoch {epoch}/{num_epochs}] [D loss: {d_loss.item()}] [G loss: {g_loss.item()}]")


[Epoch 0/200] [D loss: 0.03254091367125511] [G loss: 3.2559213638305664]
[Epoch 10/200] [D loss: 0.3247684836387634] [G loss: 0.7520647644996643]
[Epoch 20/200] [D loss: 0.31970006227493286] [G loss: 0.7747305631637573]
[Epoch 30/200] [D loss: 0.3235924243927002] [G loss: 0.7712786197662354]
[Epoch 40/200] [D loss: 0.30743417143821716] [G loss: 0.8638211488723755]
[Epoch 50/200] [D loss: 0.30076611042022705] [G loss: 0.8763408660888672]
[Epoch 60/200] [D loss: 0.3042137026786804] [G loss: 1.022635579109192]
[Epoch 70/200] [D loss: 0.46172845363616943] [G loss: 0.9596241116523743]
[Epoch 80/200] [D loss: 0.489634245634079] [G loss: 0.7327303290367126]
[Epoch 90/200] [D loss: 0.4601348340511322] [G loss: 0.9328793287277222]
[Epoch 100/200] [D loss: 0.6940193176269531] [G loss: 1.6335233449935913]
[Epoch 110/200] [D loss: 0.45026081800460815] [G loss: 1.3658318519592285]
[Epoch 120/200] [D loss: 0.433543860912323] [G loss: 0.8515220880508423]
[Epoch 130/200] [D loss: 0.3741440176963806] [

# codigo en tensorflow


In [2]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

In [3]:

output_dir = '/content/drive/MyDrive/generated_images4884'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [4]:
img_height = 250
img_width = 250
channels = 3
num_classes=53
img_shape = (img_height, img_width, channels)
latent_dim = 100
batch_size = 32  # Reducir el tamaño del batch

In [5]:
from tensorflow.keras import layers

def build_generator():
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(latent_dim + num_classes,)))
    model.add(layers.Dense(128 * (img_height // 4) * (img_width // 4), activation="relu"))
    model.add(layers.Reshape((img_height // 4, img_width // 4, 128)))
    model.add(layers.UpSampling2D(size=(2, 2)))  # Asegurar escalado consistente
    model.add(layers.Conv2D(128, kernel_size=4, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Activation("relu"))
    model.add(layers.UpSampling2D(size=(2, 2)))  # Asegurar escalado consistente
    model.add(layers.Conv2D(64, kernel_size=4, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Activation("relu"))
    model.add(layers.Conv2D(channels, kernel_size=4, padding="same"))
    model.add(layers.Activation("tanh"))
    model.add(layers.Resizing(img_height, img_width))  # Asegura que las imágenes generadas tengan el tamaño correcto
    return model

def build_discriminator():
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(img_height, img_width, channels + num_classes)))
    model.add(layers.Conv2D(64, kernel_size=4, strides=2, padding="same"))
    model.add(layers.LeakyReLU(alpha=0.2))  # Reemplaza negative_slope con alpha
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(128, kernel_size=4, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))  # Reemplaza negative_slope con alpha
    model.add(layers.Dropout(0.25))
    model.add(layers.Conv2D(256, kernel_size=4, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(alpha=0.2))  # Reemplaza negative_slope con alpha
    model.add(layers.Dropout(0.25))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

In [6]:
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        random_labels = tf.random.uniform(shape=(batch_size,), maxval=num_classes, dtype=tf.int32)
        random_labels_one_hot = tf.one_hot(random_labels, num_classes)
        random_latent_vectors_with_labels = tf.concat([random_latent_vectors, random_labels_one_hot], axis=1)

        generated_images = self.generator(random_latent_vectors_with_labels)

        # Expandir las etiquetas para que coincidan con las dimensiones espaciales de las imágenes reales
        random_labels_expanded = tf.reshape(random_labels_one_hot, (batch_size, 1, 1, num_classes))
        random_labels_tiled = tf.tile(random_labels_expanded, [1, img_height, img_width, 1])

        # Asegurarse de que las imágenes reales y generadas estén concatenadas con las etiquetas
        real_images_with_labels = tf.concat([real_images, random_labels_tiled], axis=-1)
        generated_images_with_labels = tf.concat([generated_images, random_labels_tiled], axis=-1)

        combined_images = tf.concat([generated_images_with_labels, real_images_with_labels], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(tf.concat([self.generator(random_latent_vectors_with_labels), random_labels_tiled], axis=-1))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

# Ahora, puedes usar esta clase en tu entrenamiento
cgan = CGAN()
cgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(1e-4),
    d_optimizer=tf.keras.optimizers.Adam(1e-4),
    loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=False)
)

def save_images(epoch, random_latent_vectors, output_dir="generated_images"):
    generated_images = cgan.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Reescalar a [0, 1]

    fig, axes = plt.subplots(1, batch_size, figsize=(batch_size * 2, 2))  # 1 fila y 'batch_size' columnas
    for i, img in enumerate(generated_images):
        ax = axes[i]
        ax.imshow(img.numpy())
        ax.axis('off')  # Desactivar los ejes

    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/{output_dir}/generated_img_{epoch:04d}.jpeg")
    plt.close(fig)  # Cerrar la figura para liberar memoria

def train(dataset, epochs):
    for epoch in range(epochs):
        d_loss_total = 0
        g_loss_total = 0
        for real_images in dataset:
            losses = cgan.train_step(real_images)
            d_loss_total += losses["d_loss"]
            g_loss_total += losses["g_loss"]

        d_loss_avg = d_loss_total / len(dataset)
        g_loss_avg = g_loss_total / len(dataset)

        print(f"Epoch {epoch + 1}/{epochs} | d_loss: {d_loss_avg:.4f} | g_loss: {g_loss_avg:.4f}")

        if (epoch + 1) % 10 == 0:
            random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
            save_images(epoch, random_latent_vectors)

train(dataset, epochs=200)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


NameError: name 'dataset' is not defined

In [ ]:
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        random_labels = tf.random.uniform(shape=(batch_size,), maxval=num_classes, dtype=tf.int32)
        random_labels_one_hot = tf.one_hot(random_labels, num_classes)
        random_latent_vectors_with_labels = tf.concat([random_latent_vectors, random_labels_one_hot], axis=1)

        generated_images = self.generator(random_latent_vectors_with_labels)

        # Expandir las etiquetas para que coincidan con las dimensiones espaciales de las imágenes reales
        random_labels_expanded = tf.reshape(random_labels_one_hot, (batch_size, 1, 1, num_classes))
        random_labels_tiled = tf.tile(random_labels_expanded, [1, img_height, img_width, 1])

        # Asegurarse de que las imágenes reales y generadas estén concatenadas con las etiquetas
        real_images_with_labels = tf.concat([real_images, random_labels_tiled], axis=-1)
        generated_images_with_labels = tf.concat([generated_images, random_labels_tiled], axis=-1)

        combined_images = tf.concat([generated_images_with_labels, real_images_with_labels], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors_with_labels))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}





In [ ]:
cgan = CGAN()
cgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(1e-4),
    d_optimizer=tf.keras.optimizers.Adam(1e-4),
    loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import tensorflow as tf

def save_images(epoch, random_latent_vectors, output_dir="generated_images"):
    # Genera las imágenes con el generador
    generated_images = cgan.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Reescalar a [0, 1]

    # Crear una figura grande para colocar las imágenes
    fig, axes = plt.subplots(1, batch_size, figsize=(batch_size * 2, 2))  # 1 fila y 'batch_size' columnas
    for i, img in enumerate(generated_images):
        ax = axes[i]
        ax.imshow(img.numpy())
        ax.axis('off')  # Desactivar los ejes

    # Guardar la imagen combinada en un solo archivo
    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/{output_dir}/generated_img_{epoch:04d}.jpeg")
    plt.close(fig)  # Cerrar la figura para liberar memoria


In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

new_base_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS REDIMENSIONADAS'  # Nueva ruta para imágenes redimensionadas

dataset = image_dataset_from_directory(
    new_base_dir,
    label_mode=None,
    image_size=(img_height, img_width),
    batch_size=batch_size
).map(lambda x: (x - 127.5) / 127.5)  # Normalizar imágenes a [-1, 1]

# Verificar la cantidad de archivos y clases encontradas
num_files = sum([len(files) for r, d, files in os.walk(new_base_dir)])
num_classes = len(next(os.walk(new_base_dir))[1])

print(f"Found {num_files} files in {num_classes} classes.")


Found 1051 files belonging to 1 classes.
Found 1051 files in 53 classes.


In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        d_loss_total = 0
        g_loss_total = 0
        for real_images in dataset:
            losses = cgan.train_step(real_images)
            d_loss_total += losses["d_loss"]
            g_loss_total += losses["g_loss"]

        d_loss_avg = d_loss_total / len(dataset)
        g_loss_avg = g_loss_total / len(dataset)

        print(f"Epoch {epoch + 1}/{epochs} | d_loss: {d_loss_avg:.4f} | g_loss: {g_loss_avg:.4f}")

        if (epoch + 1) % 10 == 0:
            random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
            random_labels = tf.random.uniform(shape=(batch_size,), maxval=num_classes, dtype=tf.int32)
            save_images(epoch, random_latent_vectors, random_labels)





In [ ]:
train(dataset, epochs=200)


ValueError: Input 0 of layer "sequential_7" is incompatible with the layer: expected shape=(None, 250, 250, 56), found shape=(32, 250, 250, 3)

In [ ]:
cgan

<CGAN name=cgan_1, built=False>

In [ ]:
checkpoint_dir='/content/drive/MyDrive/BASE IA/MODELOS DE IA'

In [ ]:
cgan.generator.save(f"{checkpoint_dir}/generator_epoch_.h5")
cgan.discriminator.save(f"{checkpoint_dir}/discriminator_epoch_.h5")
print(f"Modelos guardados para la época")

AttributeError: module 'tensorflow.keras' has no attribute 'saving'

# MODELO TESNOR MODIFICADO

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# Configuración para trabajar con imágenes en blanco y negro
output_dir = '/content/drive/MyDrive/generated_imagesNoemi'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

img_height = 250
img_width = 250
channels = 1  # Blanco y negro (1 canal)
img_shape = (img_height, img_width, channels)
latent_dim = 100
batch_size = 32  # Reducir el tamaño del batch

# Ajuste de hiperparámetros
learning_rate = 1e-4
beta_1 = 0.5  # Para optimizadores Adam

In [ ]:
# Construcción del generador
def build_generator():
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(latent_dim,)))
    model.add(layers.Dense(128 * (img_height // 4) * (img_width // 4), activation="relu"))
    model.add(layers.Reshape((img_height // 4, img_width // 4, 128)))
    model.add(layers.UpSampling2D(size=(2, 2)))
    model.add(layers.Conv2D(128, kernel_size=4, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Activation("relu"))
    model.add(layers.UpSampling2D(size=(2, 2)))
    model.add(layers.Conv2D(64, kernel_size=4, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Activation("relu"))
    model.add(layers.Conv2D(channels, kernel_size=4, padding="same"))
    model.add(layers.Activation("tanh"))
    model.add(layers.Resizing(img_height, img_width))
    return model


In [ ]:
# Construcción del discriminador
def build_discriminator():
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=img_shape))
    model.add(layers.Conv2D(64, kernel_size=4, strides=2, padding="same"))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))  # Incremento de Dropout para evitar sobreajuste
    model.add(layers.Conv2D(128, kernel_size=4, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))
    model.add(layers.Conv2D(256, kernel_size=4, strides=2, padding="same"))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dropout(0.3))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

In [ ]:
# Definición del modelo CGAN
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.05 * tf.random.uniform(tf.shape(labels))  # Suavizado de etiquetas

        # Entrenamiento del discriminador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        # Entrenamiento del generador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}


In [ ]:
# Definición del modelo CGAN
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.05 * tf.random.uniform(tf.shape(labels))  # Suavizado de etiquetas

        # Entrenamiento del discriminador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        # Entrenamiento del generador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

# Configuración del modelo CGAN
cgan = CGAN()
cgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    d_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=False)
)

In [ ]:
def save_images(epoch, random_latent_vectors, output_dir="generated_imagesNoemi"):
    generated_images = cgan.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Reescalar a [0, 1]

    fig, axes = plt.subplots(1, min(batch_size, 8), figsize=(8 * 2, 2))
    for i, img in enumerate(generated_images[:8]):
        ax = axes[i]
        ax.imshow(img.numpy().squeeze(), cmap="gray")  # Mostrar en escala de grises
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/{output_dir}/generated_img_{epoch:04d}.jpeg")
    plt.close(fig)

from tensorflow.keras.preprocessing import image_dataset_from_directory

new_base_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS REDIMENSIONADAS'

dataset = image_dataset_from_directory(
    new_base_dir,
    label_mode=None,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode="grayscale"  # Cargar imágenes en blanco y negro
).map(lambda x: (x - 127.5) / 127.5)  # Normalizar imágenes a [-1, 1]

# Verificar la cantidad de archivos y clases encontradas
num_files = sum([len(files) for r, d, files in os.walk(new_base_dir)])
num_classes = len(next(os.walk(new_base_dir))[1])
print(f"Found {num_files} files in {num_classes} classes.")

Found 1051 files.
Found 1051 files in 53 classes.


In [ ]:
# Función de entrenamiento con visualización más frecuente
def train(dataset, epochs):
    for epoch in range(epochs):
        d_loss_total = 0
        g_loss_total = 0
        for real_images in dataset:
            losses = cgan.train_step(real_images)
            d_loss_total += losses["d_loss"]
            g_loss_total += losses["g_loss"]

        d_loss_avg = d_loss_total / len(dataset)
        g_loss_avg = g_loss_total / len(dataset)

        print(f"Epoch {epoch + 1}/{epochs} | d_loss: {d_loss_avg:.4f} | g_loss: {g_loss_avg:.4f}")

        if (epoch + 1) % 5 == 0:  # Guardar imágenes cada 5 épocas
            random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
            save_images(epoch, random_latent_vectors)

In [ ]:
train(dataset, epochs=400)

Epoch 1/400 | d_loss: 0.3495 | g_loss: 1.0278
Epoch 2/400 | d_loss: 0.4847 | g_loss: 1.3541
Epoch 3/400 | d_loss: 0.4407 | g_loss: 1.4298
Epoch 4/400 | d_loss: 0.2633 | g_loss: 1.8497
Epoch 5/400 | d_loss: 0.1852 | g_loss: 2.3188
Epoch 6/400 | d_loss: 0.2916 | g_loss: 2.5111
Epoch 7/400 | d_loss: 0.1259 | g_loss: 2.4860
Epoch 8/400 | d_loss: 0.0979 | g_loss: 3.3167
Epoch 9/400 | d_loss: -0.0608 | g_loss: 3.0451
Epoch 10/400 | d_loss: 0.0404 | g_loss: 4.2967
Epoch 11/400 | d_loss: 0.0330 | g_loss: 3.9652
Epoch 12/400 | d_loss: 0.6832 | g_loss: 5.1721
Epoch 13/400 | d_loss: 1.0591 | g_loss: 2.0424
Epoch 14/400 | d_loss: 0.4567 | g_loss: 1.0247
Epoch 15/400 | d_loss: 0.3416 | g_loss: 1.5408
Epoch 16/400 | d_loss: 0.2872 | g_loss: 1.5664
Epoch 17/400 | d_loss: 0.4870 | g_loss: 2.0153
Epoch 18/400 | d_loss: 0.6161 | g_loss: 2.0887
Epoch 19/400 | d_loss: 0.2498 | g_loss: 1.7894
Epoch 20/400 | d_loss: 0.2578 | g_loss: 1.6868
Epoch 21/400 | d_loss: 0.2429 | g_loss: 1.8626
Epoch 22/400 | d_loss

# ron y san codigo

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# Configuración para trabajar con imágenes en blanco y negro
output_dir = '/content/drive/MyDrive/generated_images3'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

img_height = 250
img_width = 250
channels = 1  # Blanco y negro (1 canal)
img_shape = (img_height, img_width, channels)
latent_dim = 100
batch_size = 32

# Ajuste de hiperparámetros
learning_rate = 1e-5  # Reducida tasa de aprendizaje
beta_1 = 0.5  # Para optimizadores Adam
patience = 15  # Parada temprana ajustada para mayor paciencia



In [ ]:

# Construcción del generador con más filtros y más capas
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(256 * (img_height // 4) * (img_width // 4), activation="relu"),
        layers.Reshape((img_height // 4, img_width // 4, 256)),
        layers.UpSampling2D(size=(2, 2)),
        layers.Conv2D(256, kernel_size=4, padding="same", activation="relu"),
        layers.UpSampling2D(size=(2, 2)),
        layers.Conv2D(128, kernel_size=4, padding="same", activation="relu"),
        layers.Conv2D(64, kernel_size=4, padding="same", activation="relu"),
        layers.Conv2D(channels, kernel_size=4, padding="same", activation="tanh"),
        layers.Resizing(img_height, img_width)
    ])
    return model



In [ ]:
# Construcción del discriminador más fuerte
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=img_shape),
        layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ])
    return model



In [ ]:
# Definición del modelo CGAN
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.1 * tf.random.uniform(tf.shape(labels))  # Suavizado de etiquetas

        # Entrenamiento del discriminador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        # Entrenamiento del generador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

# Configuración del modelo CGAN
cgan = CGAN()
cgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    d_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=False)
)



In [ ]:
# Función para guardar imágenes generadas
def save_images(epoch, random_latent_vectors, output_dir="generated_images3"):
    generated_images = cgan.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Reescalar a [0, 1]

    fig, axes = plt.subplots(1, min(batch_size, 8), figsize=(8 * 2, 2))
    for i, img in enumerate(generated_images[:8]):
        ax = axes[i]
        ax.imshow(img.numpy().squeeze(), cmap="gray")
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/{output_dir}/generated_img_{epoch:04d}.jpeg")
    plt.close(fig)

# Cargar dataset en blanco y negro
from tensorflow.keras.preprocessing import image_dataset_from_directory

new_base_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS REDIMENSIONADAS'

dataset = image_dataset_from_directory(
    new_base_dir,
    label_mode=None,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode="grayscale"  # Cargar imágenes en blanco y negro
).map(lambda x: (x - 127.5) / 127.5)  # Normalizar imágenes a [-1, 1]

# Verificar la cantidad de archivos y clases encontradas
num_files = sum([len(files) for r, d, files in os.walk(new_base_dir)])
num_classes = len(next(os.walk(new_base_dir))[1])
print(f"Found {num_files} files in {num_classes} classes.")



Found 1051 files.
Found 1051 files in 53 classes.


In [ ]:
# Función de entrenamiento con parada temprana y visualización más frecuente
def train(dataset, epochs, patience):
    best_g_loss = float('inf')
    wait = 0
    min_g_loss = 10  # Umbral de cambio de g_loss para early stopping

    for epoch in range(epochs):
        d_loss_total = tf.keras.metrics.Mean()
        g_loss_total = tf.keras.metrics.Mean()

        for real_images in dataset:
            losses = cgan.train_step(real_images)
            d_loss_total.update_state(losses["d_loss"])
            g_loss_total.update_state(losses["g_loss"])

        print(f"Epoch {epoch + 1}/{epochs} | d_loss: {d_loss_total.result():.4f} | g_loss: {g_loss_total.result():.4f}")

        # Guardar imágenes generadas cada 2 épocas
        if (epoch + 1) % 10 == 0:
            random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
            save_images(epoch, random_latent_vectors)

        # Early stopping modificado
        if g_loss_total.result() < min_g_loss:
            min_g_loss = g_loss_total.result()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break


In [ ]:
train(dataset, epochs=200, patience=10)

Epoch 1/200 | d_loss: 0.5037 | g_loss: 0.6267
Epoch 2/200 | d_loss: 0.5879 | g_loss: 0.4273
Epoch 3/200 | d_loss: 0.5691 | g_loss: 0.6152
Epoch 4/200 | d_loss: 0.4275 | g_loss: 0.7512
Epoch 5/200 | d_loss: 0.5821 | g_loss: 0.5530
Epoch 6/200 | d_loss: 0.5270 | g_loss: 0.6642
Epoch 7/200 | d_loss: 0.3955 | g_loss: 0.8997
Epoch 8/200 | d_loss: 0.4542 | g_loss: 0.5984
Epoch 9/200 | d_loss: 0.4667 | g_loss: 0.6550
Epoch 10/200 | d_loss: 0.4954 | g_loss: 0.7573
Epoch 11/200 | d_loss: 0.4872 | g_loss: 0.8289
Epoch 12/200 | d_loss: 0.4693 | g_loss: 0.8990
Early stopping triggered.


# NUEVO METODO NUEVOO

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# Configuración para trabajar con imágenes en blanco y negro
output_dir = '/content/drive/MyDrive/generated_images3'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

img_height = 250
img_width = 250
channels = 1  # Blanco y negro (1 canal)
img_shape = (img_height, img_width, channels)
latent_dim = 100
batch_size = 32

# Ajuste de hiperparámetros
learning_rate = 1e-4  # Ajustada la tasa de aprendizaje
beta_1 = 0.5  # Para optimizadores Adam
patience = 15  # Parada temprana ajustada para mayor paciencia



In [ ]:
# Construcción del generador con más filtros y más capas
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(256 * (img_height // 4) * (img_width // 4), activation="relu"),
        layers.Reshape((img_height // 4, img_width // 4, 256)),
        layers.UpSampling2D(size=(2, 2)),
        layers.Conv2D(256, kernel_size=4, padding="same", activation="relu"),
        layers.UpSampling2D(size=(2, 2)),
        layers.Conv2D(128, kernel_size=4, padding="same", activation="relu"),
        layers.Conv2D(64, kernel_size=4, padding="same", activation="relu"),
        layers.Conv2D(channels, kernel_size=4, padding="same", activation="tanh"),
        layers.Resizing(img_height, img_width)
    ])
    return model

# Construcción del discriminador más fuerte
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=img_shape),
        layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.4),
        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ])
    return model




In [ ]:
# Definición del modelo CGAN
class CGAN(tf.keras.Model):
    def __init__(self):
        super(CGAN, self).__init__()
        self.generator = build_generator()
        self.discriminator = build_discriminator()

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(CGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        labels += 0.1 * tf.random.uniform(tf.shape(labels))  # Suavizado de etiquetas

        # Entrenamiento del discriminador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        misleading_labels = tf.ones((batch_size, 1))

        # Entrenamiento del generador
        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

# Configuración del modelo CGAN
cgan = CGAN()
cgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    d_optimizer=tf.keras.optimizers.Adam(learning_rate, beta_1=beta_1),
    loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=False)
)



In [ ]:
# Función para guardar imágenes generadas
def save_images(epoch, random_latent_vectors, output_dir="generated_images3"):
    generated_images = cgan.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Reescalar a [0, 1]

    fig, axes = plt.subplots(1, min(batch_size, 8), figsize=(8 * 2, 2))
    for i, img in enumerate(generated_images[:8]):
        ax = axes[i]
        ax.imshow(img.numpy().squeeze(), cmap="gray")
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/{output_dir}/generated_img_{epoch:04d}.jpeg")
    plt.close(fig)

# Cargar dataset en blanco y negro
from tensorflow.keras.preprocessing import image_dataset_from_directory

new_base_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS REDIMENSIONADAS'

dataset = image_dataset_from_directory(
    new_base_dir,
    label_mode=None,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    color_mode="grayscale"  # Cargar imágenes en blanco y negro
).map(lambda x: (x - 127.5) / 127.5)  # Normalizar imágenes a [-1, 1]

# Verificar la cantidad de archivos y clases encontradas
num_files = sum([len(files) for r, d, files in os.walk(new_base_dir)])
num_classes = len(next(os.walk(new_base_dir))[1])
print(f"Found {num_files} files in {num_classes} classes.")




Found 1051 files.
Found 1051 files in 53 classes.


In [ ]:
# Función de entrenamiento con parada temprana y visualización más frecuente
def train(dataset, epochs, patience):
    best_g_loss = float('inf')
    wait = 0
    min_g_loss = 10  # Umbral de cambio de g_loss para early stopping

    for epoch in range(epochs):
        d_loss_total = tf.keras.metrics.Mean()
        g_loss_total = tf.keras.metrics.Mean()

        for real_images in dataset:
            losses = cgan.train_step(real_images)
            d_loss_total.update_state(losses["d_loss"])
            g_loss_total.update_state(losses["g_loss"])

        print(f"Epoch {epoch + 1}/{epochs} | d_loss: {d_loss_total.result():.4f} | g_loss: {g_loss_total.result():.4f}")

        # Guardar imágenes generadas cada 2 épocas
        if (epoch + 1) % 2 == 0:
            random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))
            save_images(epoch, random_latent_vectors)

        # Early stopping modificado
        if g_loss_total.result() < min_g_loss:
            min_g_loss = g_loss_total.result()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break


In [ ]:
train(dataset, epochs=200, patience=15)

Epoch 1/200 | d_loss: 0.5569 | g_loss: 0.6451
Epoch 2/200 | d_loss: 0.4701 | g_loss: 1.1844
Epoch 3/200 | d_loss: 0.3760 | g_loss: 1.3483
Epoch 4/200 | d_loss: 0.2580 | g_loss: 1.8355
Epoch 5/200 | d_loss: 0.7654 | g_loss: 1.5668
Epoch 6/200 | d_loss: 0.5154 | g_loss: 1.0912
Epoch 7/200 | d_loss: 0.6802 | g_loss: 0.9089
Epoch 8/200 | d_loss: 0.5237 | g_loss: 1.0102
Epoch 9/200 | d_loss: 0.6223 | g_loss: 1.0547
Epoch 10/200 | d_loss: 0.5360 | g_loss: 1.0038
Epoch 11/200 | d_loss: 0.5087 | g_loss: 1.1921
Epoch 12/200 | d_loss: 0.4638 | g_loss: 1.1231
Epoch 13/200 | d_loss: 0.5624 | g_loss: 1.1342
Epoch 14/200 | d_loss: 0.5100 | g_loss: 1.3234
Epoch 15/200 | d_loss: 0.3743 | g_loss: 1.3016
Epoch 16/200 | d_loss: 0.5957 | g_loss: 1.5178
Early stopping triggered.


# wgan entrando

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Parámetros principales
img_shape = (250, 250, 1)  # Dimensiones de las imágenes
latent_dim = 100          # Dimensión del espacio latente

def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(32 * 32 * 256, input_dim=latent_dim),
        layers.Reshape((32, 32, 256)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),  # Tamaño: 64x64
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),   # Tamaño: 128x128
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(32, kernel_size=4, strides=2, padding="same"),   # Tamaño: 256x256
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(1, kernel_size=7, activation="tanh", padding="same")      # Tamaño final: 250x250
    ])
    return model


def build_discriminator():
    model = tf.keras.Sequential([
      layers.Input(shape=(250, 250, 1)),  # Asegúrate de que las dimensiones sean correctas aquí
      layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
      layers.LeakyReLU(0.2),
      layers.Dropout(0.3),
      layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
      layers.LeakyReLU(0.2),
      layers.Dropout(0.3),
      layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),
      layers.LeakyReLU(0.2),
      layers.Dropout(0.3),
      layers.Conv2D(512, kernel_size=4, strides=2, padding="same"),
      layers.LeakyReLU(0.2),
      layers.Dropout(0.3),
      layers.Flatten(),
      layers.Dense(1)  # Sin activación para WGAN
    ])
    return model




In [ ]:
def gradient_penalty(discriminator, real_images, fake_images):
    """Calcula el gradient penalty para WGAN-GP."""
    # Obtener el batch size
    batch_size = tf.shape(real_images)[0]

    # Asegurarse de que ambas imágenes tengan la misma forma
    fake_images = tf.image.resize(fake_images, (250, 250))  # Redimensionar a 250x250
    real_images = tf.image.resize(real_images, (250, 250))  # Redimensionar a 250x250

    epsilon = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
    interpolated = epsilon * real_images + (1 - epsilon) * fake_images

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        interpolated_predictions = discriminator(interpolated, training=True)  # Asegúrate de pasar training=True
    gradients = tape.gradient(interpolated_predictions, [interpolated])[0]
    gradients_sqr = tf.square(gradients)
    gradients_sqr_sum = tf.reduce_sum(gradients_sqr, axis=[1, 2, 3])
    gradient_l2_norm = tf.sqrt(gradients_sqr_sum)
    penalty = tf.reduce_mean((gradient_l2_norm - 1.0) ** 2)
    return penalty




def wasserstein_loss(y_true, y_pred):
    return tf.reduce_mean(y_true * y_pred)



In [ ]:
class WGAN(tf.keras.Model):
    def __init__(self, generator, discriminator, gp_weight=10.0):
        super(WGAN, self).__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.gp_weight = gp_weight

    def compile(self, g_optimizer, d_optimizer):
        super(WGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer

    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        random_latent_vectors = tf.random.normal(shape=(batch_size, latent_dim))

        # Entrenamiento del discriminador
        with tf.GradientTape() as tape:
            fake_images = self.generator(random_latent_vectors, training=True)
            real_logits = self.discriminator(real_images, training=True)
            fake_logits = self.discriminator(fake_images, training=True)

            d_loss_real = wasserstein_loss(tf.ones_like(real_logits), real_logits)
            d_loss_fake = wasserstein_loss(-tf.ones_like(fake_logits), fake_logits)

            gp = gradient_penalty(self.discriminator, real_images, fake_images)
            d_loss = d_loss_real + d_loss_fake + self.gp_weight * gp

        d_grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_weights))

        # Entrenamiento del generador
        misleading_labels = tf.ones((batch_size, 1))
        with tf.GradientTape() as tape:
            fake_images = self.generator(random_latent_vectors, training=True)
            fake_logits = self.discriminator(fake_images, training=True)
            g_loss = wasserstein_loss(misleading_labels, fake_logits)

        g_grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

# Crear el generador y discriminador con las dimensiones correctas
generator = build_generator()
discriminator = build_discriminator()

# Verificar que las imágenes generadas tienen la forma correcta
fake_images = generator(tf.random.normal([batch_size, latent_dim]))
print(fake_images.shape)  # Debería mostrar (batch_size, 250, 250, 1)

# Verificar las imágenes reales que se pasan al discriminador
real_images, _ = next(dataset)  # Tomar un batch de imágenes reales
print(real_images.shape)  # Debería mostrar (batch_size, 250, 250, 1)


# Instanciar el modelo WGAN
wgan = WGAN(generator, discriminator)

# Compilación del modelo
wgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9),
    d_optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
)



(32, 256, 256, 1)


NameError: name 'dataset' is not defined

In [ ]:
# Cargar y preparar los datos locales
base_dir = '/content/drive/MyDrive/BASE DE ESQUEMAS REDIMENSIONADAS'
data_gen = ImageDataGenerator(rescale=1.0 / 127.5 - 1)  # Normalizar a rango [-1, 1]
dataset = data_gen.flow_from_directory(
    base_dir,
    target_size=(250, 250),
    color_mode="grayscale",
    batch_size=64,
    class_mode=None
)



Found 1051 images belonging to 53 classes.


In [ ]:
# Entrenamiento del modelo
history = wgan.fit(dataset, epochs=200)

# Guardar el generador entrenado
generator.save('wgan_generator.h5')

# Función para mostrar imágenes generadas durante el entrenamiento
def generate_images(model, epoch, latent_dim=100, num_images=10):
    random_latent_vectors = tf.random.normal(shape=(num_images, latent_dim))
    generated_images = model.generator(random_latent_vectors)
    generated_images = (generated_images + 1) / 2.0  # Desnormalizar a rango [0, 1]

    fig, axes = plt.subplots(1, num_images, figsize=(20, 4))
    for i in range(num_images):
        axes[i].imshow(generated_images[i, :, :, 0], cmap='gray')
        axes[i].axis('off')
    plt.show()

# Mostrar algunas imágenes generadas durante el entrenamiento
generate_images(wgan, epoch=0)

Epoch 1/200


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "conv2d_11" is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (1, None, 250, 250)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, None, 250, 250, 1), dtype=float32)
  • training=None
  • mask=None